# Run All — Full Setup from Scratch

This notebook **cleans up all existing resources** and then **re-creates everything
from scratch** by running each notebook in the correct order.

Uses `dbutils.notebook.run()` so each notebook executes in its own isolated context.

### Execution Order

| # | Notebook | Purpose | Est. Time |
|---|----------|---------|-----------|
| 0 | `_helper/cleanup_all` | Delete app, Lakebase project, Delta tables, schema | \~3 min |
| 1 | `_helper/01_generate_sap_data` | Generate SAP EKKO & EKPO tables in Delta | \~1 min |
| 2 | `_helper/02_generate_oltp_data` | Generate dock_slot & delivery_booking in Delta | \~1 min |
| 3 | `01_SAP_Data_Pipeline` | Join EKKO + EKPO → ekpo_enriched | \~1 min |
| 4 | `02_Lakebase_Setup` | Provision Lakebase, create tables, load data | \~5 min |
| 5 | `04_App_Deployment` | Deploy Databricks App | \~5 min |

**Total estimated time: \~15 minutes**

In [0]:
import time
import re

# Notebooks to run in order (paths relative to this notebook's location)
NOTEBOOKS = [
    ("cleanup_all",              "Cleanup all existing resources"),
    ("01_generate_sap_data",     "Generate SAP EKKO & EKPO data"),
    ("02_generate_oltp_data",    "Generate dock_slot & delivery_booking data"),
    ("../01_SAP_Data_Pipeline",  "Create ekpo_enriched table"),
    ("../02_Lakebase_Setup",     "Provision Lakebase and load data"),
    ("../04_App_Deployment",     "Deploy Databricks App"),
]

TIMEOUT = 1200  # 20 minutes per notebook
MAX_RETRIES = 2  # Retry failed notebooks once (handles transient issues)

results = []
print("=" * 70)
print("  FULL SETUP — Starting end-to-end run")
print("=" * 70)

for i, (path, description) in enumerate(NOTEBOOKS):
    name = path.split("/")[-1]
    print(f"\n{'─' * 70}")
    print(f"  [{i}/{len(NOTEBOOKS)-1}] {description}")
    print(f"       Notebook: {name}")
    print(f"{'─' * 70}")

    # Add stabilization pause before App Deployment
    if name == "04_App_Deployment":
        print("  Waiting 15s for Lakebase to stabilize...")
        time.sleep(15)

    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        start = time.time()
        try:
            result = dbutils.notebook.run(path, TIMEOUT)
            elapsed = time.time() - start
            results.append((name, "✓", f"{elapsed:.0f}s", ""))
            print(f"  ✓ Completed in {elapsed:.0f}s")
            last_error = None
            break
        except Exception as e:
            elapsed = time.time() - start
            match = re.search(r'FAILED:\s*(.+?)(?:\n|$)', str(e))
            last_error = match.group(1).strip() if match else str(e)[:300]
            if attempt < MAX_RETRIES:
                print(f"  ⚠ Attempt {attempt} failed after {elapsed:.0f}s: {last_error}")
                print(f"  Retrying in 10s...")
                time.sleep(10)
            else:
                results.append((name, "✗", f"{elapsed:.0f}s", last_error))
                print(f"  ✗ FAILED after {elapsed:.0f}s (attempt {attempt}/{MAX_RETRIES})")
                print(f"  Error: {last_error}")

    if last_error:
        print(f"  ⚠ Stopping — fix the issue and re-run.")
        break

# ── Summary ──
print(f"\n\n{'=' * 70}")
print("  SUMMARY")
print(f"{'=' * 70}")
print(f"{'Notebook':<30} {'Status':<6} {'Time':<8} {'Details'}")
print(f"{'─' * 70}")
for name, status, elapsed, error in results:
    detail = error if error else ""
    print(f"{name:<30} {status:<6} {elapsed:<8} {detail[:60]}")

all_passed = all(s == '✓' for _, s, _, _ in results) and len(results) == len(NOTEBOOKS)
print(f"\n{'✓ All notebooks completed successfully!' if all_passed else '✗ Some notebooks failed — see above.'}")

if all_passed:
    print(f"\n  App URL: https://delivery-slot-booking-7474652078742031.aws.databricksapps.com")